# Snapshot Generation and Evaluation

This notebook reconstructs the public forecast sequence as a set of strictly cutoff-aware snapshots. It calls the project's own `train_model_up_to()`, `BayesianDixonColesModel`, and `BayesianWorldCup2026` code; it does not reimplement the model. All result products are written below `paper_analysis/`.

**Mode policy.** `PLAN` is the default and does not read data or create artifacts. `FAST` runs an explicit no-MCMC input and dependency preflight but creates no posterior or forecast output. `FULL` samples the original Stan model and simulates the tournament, then checkpoints each completed snapshot. No cell fabricates a probability, score, parameter estimate, or realised match result.

## Reproducibility protocol

The snapshot plan is read from the committed public-site labels and calendar. The cutoffs use the final scheduled date of each completed round, rather than a same-day display label, because `results.csv` has dates but not kick-off timestamps. A match is evaluated with the latest snapshot dated *strictly before* its match date.

The default FULL configuration uses 100,000 tournament simulations and the project's fixed seed. The original sampler configuration is recorded in every `run_metadata.json`. The final pre-final snapshot is compared with `docs/csv/previsoes/all_matchups.csv` in percentage points against an explicit tolerance.

In [1]:
# Configuration: run from the repository root or paper_analysis/.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
ANALYSIS_ROOT = PROJECT_ROOT / 'analysis'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODE = 'FULL'  # Set FAST for an explicit preflight or FULL for the full analysis.
N_SIMULATIONS = 100_000
SIMULATION_SEED = 42
FORCE_RETRAIN = False
REFERENCE_TOLERANCE_PCT_POINTS = 0.05
RUN_BACKTESTS = False  # validate.py is expensive and requires data/raw/results.csv.
N_BOOTSTRAP = 2_000

assert MODE in {'PLAN', 'FAST', 'FULL'}
from paper_pipeline import (
    actual_world_cup_results, backtest_preflight, evaluate_from_saved,
    run_all_snapshots, run_fast_preflight, run_project_backtests, snapshot_plan
)

try:
    from IPython.display import display
except ImportError:
    display = print

## A. Input audit and phase cutoffs

In [2]:
# This preflight runs only after an explicit FAST or FULL selection.
if MODE in {'FAST', 'FULL'}:
    preflight_report = run_fast_preflight(PROJECT_ROOT)
    display(snapshot_plan(PROJECT_ROOT))
    preflight_report
else:
    print('PLAN mode: no source audit or files were generated.')

Dados preparados: 3387 jogos e 190 seleções identificadas.


,snapshot_order,snapshot_id,public_version_label,cutoff_date,phase,cutoff_basis,note,source_site_file,source_schedule_file
0,1,pre_fifa,Antes da data FIFA,2026-05-28,Pre-tournament,Day immediately before the FIFA-window boundar...,The public site distinguishes this earlier pre...,docs/js/chances.js,docs/csv/vis/calendario.csv
1,2,pre_tournament_post_fifa,Antes da Copa - pós data FIFA,2026-06-10,Pre-tournament,Day before the opening match in the committed ...,Primary pre-tournament comparator in the evalu...,docs/js/chances.js,docs/csv/vis/calendario.csv
2,3,after_group_round_1,Após 1ª rodada da fase de grupos,2026-06-17,"Group stage, round 1 complete",Last scheduled date of group-stage round 1 in ...,The code's display boundary is one day later; ...,docs/js/chances.js,docs/csv/vis/calendario.csv
3,4,after_group_round_2,Após 2ª rodada da fase de grupos,2026-06-23,"Group stage, round 2 complete",Last scheduled date of group-stage round 2 in ...,The code's display boundary is one day later; ...,docs/js/chances.js,docs/csv/vis/calendario.csv
4,5,after_group_stage,Após a Fase de Grupos,2026-06-27,Group stage complete,Last scheduled group-stage date in calendario....,All group-stage results on or before this date...,docs/js/chances.js,docs/csv/vis/calendario.csv
5,6,after_round_of_32,Após os 16-Avos,2026-07-03,Round of 32 complete,Last scheduled round-of-32 date in calendario....,The next knockout phase is simulated condition...,docs/js/chances.js,docs/csv/vis/calendario.csv
6,7,after_round_of_16,Após as Oitavas,2026-07-07,Round of 16 complete,Last scheduled round-of-16 date in calendario....,The next knockout phase is simulated condition...,docs/js/chances.js,docs/csv/vis/calendario.csv
7,8,after_quarterfinals,Após as Quartas,2026-07-11,Quarter-finals complete,Last scheduled quarter-final date in calendari...,The next knockout phase is simulated condition...,docs/js/chances.js,docs/csv/vis/calendario.csv
8,9,after_semifinals,Após as Semifinais,2026-07-15,Semi-finals complete,Last scheduled semi-final date in calendario.csv.,This is the final pre-final snapshot and the r...,docs/js/chances.js,docs/csv/vis/calendario.csv


The optional report above is a **preflight record**, not a model result. It records the status of CmdStan, the Parquet engine, source-file schemas, hashes, planned cutoffs, and whether every supplied 2026 score is available.

In [3]:
# Inspect observed data completeness only in an explicit execution mode.
if MODE in {'FAST', 'FULL'}:
    realized_results = actual_world_cup_results(PROJECT_ROOT)
    display(realized_results[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'score_observed', 'calendar_phase']])
    print('Observed scores:', int(realized_results['score_observed'].sum()))
    print('Rows with a missing score:', int((~realized_results['score_observed']).sum()))
else:
    print('PLAN mode: results completeness was not read.')

,date,home_team,away_team,home_score,away_score,score_observed,calendar_phase
0,2026-06-11,Mexico,South Africa,2,0,True,Fase de grupos
1,2026-06-11,South Korea,Czech Republic,2,1,True,Fase de grupos
2,2026-06-12,Canada,Bosnia and Herzegovina,1,1,True,Fase de grupos
3,2026-06-12,United States,Paraguay,4,1,True,Fase de grupos
4,2026-06-13,Australia,Turkey,2,0,True,Fase de grupos
...,...,...,...,...,...,...,...
99,2026-07-11,Norway,England,1,1,True,Quartas de final
100,2026-07-14,France,Spain,0,2,True,Semifinal
101,2026-07-15,England,Argentina,1,2,True,Semifinal
102,2026-07-18,France,England,4,6,True,Disputa de 3º lugar


Observed scores: 104
Rows with a missing score: 0


## B. FULL snapshot generation

In [4]:
# This branch fits only data on or before each cutoff, then fixes only known World Cup results at that cutoff.
if MODE == 'FULL':
    generation_manifest = run_all_snapshots(
        PROJECT_ROOT,
        n_sim=N_SIMULATIONS,
        seed=SIMULATION_SEED,
        force_retrain=FORCE_RETRAIN,
        tolerance_pct_points=REFERENCE_TOLERANCE_PCT_POINTS,
    )
    display(generation_manifest)
else:
    print(f'{MODE} mode: no posterior sampling or forecast tables were created.')

Modelo já treinado para 2026-05-28, reaproveitando: data/outputs/models/draws_2026_2026-05-28_n_poisson_ranking.npz
Dados preparados: 3102 jogos e 187 seleções identificadas.
Modelo já treinado para 2026-06-10, reaproveitando: data/outputs/models/draws_2026_2026-06-10_n_poisson_ranking.npz
Dados preparados: 3285 jogos e 190 seleções identificadas.
Modelo já treinado para 2026-06-17, reaproveitando: data/outputs/models/draws_2026_2026-06-17_n_poisson_ranking.npz
Dados preparados: 3309 jogos e 190 seleções identificadas.
Modelo já treinado para 2026-06-23, reaproveitando: data/outputs/models/draws_2026_2026-06-23_n_poisson_ranking.npz
Dados preparados: 3333 jogos e 190 seleções identificadas.
Modelo já treinado para 2026-06-27, reaproveitando: data/outputs/models/draws_2026_2026-06-27_n_poisson_ranking.npz
Dados preparados: 3357 jogos e 190 seleções identificadas.
Modelo já treinado para 2026-07-03, reaproveitando: data/outputs/models/draws_2026_2026-07-03_n_poisson_ranking.npz
Dados pre

,snapshot_id,status,directory,runtime_seconds
0,pre_fifa,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,53.647877
1,pre_tournament_post_fifa,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,49.903308
2,after_group_round_1,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,54.994094
3,after_group_round_2,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,54.610573
4,after_group_stage,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,58.773164
5,after_round_of_32,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,53.501245
6,after_round_of_16,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,51.972197
7,after_quarterfinals,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,54.736084
8,after_semifinals,generated,/home/iara/Github/WorldCup2026/analysis/artifa...,52.954965


Each FULL snapshot saves the following tidy products: all-matchup 0--4 scoreline cells plus residual mass, 1X2 and expected-goal summaries, goal marginals for PIT, stage probabilities with MCSE, the deterministic projected bracket, posterior summaries, observed-result completeness, metadata, and (at the final snapshot) the reference comparison.

## C. Evaluation from saved products

In [ ]:
# Evaluation is deliberately separated from fitting and reads saved Parquet products only.
if MODE == 'FULL':
    evaluation_products = evaluate_from_saved(
        PROJECT_ROOT, n_bootstrap=N_BOOTSTRAP, seed=20261014
    )
    display(evaluation_products['match_level_metrics'])
    display(evaluation_products['tournament_level_metrics'])
    display(evaluation_products['paired_bootstrap'])
else:
    print(f'{MODE} mode: evaluation waits for saved FULL snapshot products.')

Dados preparados: 3285 jogos e 190 seleções identificadas.
Sucesso! Posições do ranking /home/iara/Github/WorldCup2026/data/raw/fifa_ranking_2022.csv carregadas e normalizadas.
Dados preparados: 3309 jogos e 190 seleções identificadas.
Sucesso! Posições do ranking /home/iara/Github/WorldCup2026/data/raw/fifa_ranking_2022.csv carregadas e normalizadas.
Dados preparados: 3333 jogos e 190 seleções identificadas.
Sucesso! Posições do ranking /home/iara/Github/WorldCup2026/data/raw/fifa_ranking_2022.csv carregadas e normalizadas.
Dados preparados: 3357 jogos e 190 seleções identificadas.
Sucesso! Posições do ranking /home/iara/Github/WorldCup2026/data/raw/fifa_ranking_2022.csv carregadas e normalizadas.
Dados preparados: 3373 jogos e 190 seleções identificadas.
Sucesso! Posições do ranking /home/iara/Github/WorldCup2026/data/raw/fifa_ranking_2022.csv carregadas e normalizadas.
Dados preparados: 3381 jogos e 190 seleções identificadas.
Sucesso! Posições do ranking /home/iara/Github/WorldCup2

The evaluation includes multiclass Brier, ordered RPS, log score and skill scores against uniform, training-frequency and static-FIFA-ranking logistic baselines; as-of versus pre-tournament comparisons; calibration and draw diagnostics; censored scoreline log score, expected-goal errors, randomized PIT and dispersion checks; stage and bracket metrics; information change across snapshots; and paired bootstrap intervals.

The scoreline log score is exact inside the published 0--4 grid. For a realised score outside it, the saved residual mass defines a clearly labelled censored category rather than inventing an unexported point probability.

## D. Historical backtest feasibility

In [ ]:
if MODE == 'FULL':
    backtest_status = backtest_preflight(PROJECT_ROOT)
    display(backtest_status)
    if RUN_BACKTESTS:
        display(run_project_backtests(PROJECT_ROOT))
    else:
        print('Backtests remain disabled by configuration.')
else:
    print(f'{MODE} mode: backtest preflight was not run.')

,component,required_file,available,status,detail
0,src.model_sel.validate,data/raw/results.csv,False,blocked,The project script hard-codes data/raw/results...
1,jogos_2014/2018/2022,data/raw/jogos_2014.csv; data/raw/jogos_2018.c...,True,available,"These three test-set files are present, but va..."


Backtests remain disabled by configuration.


`validate.py` is invoked only when its documented full historical input exists. The notebook does not substitute a different training file or manufacture backtest results. Interpret all score comparisons cautiously: a single World Cup supplies few matches, and bootstrap intervals are descriptive rather than confirmatory.